# 🔱 VoiceBatch Studio v2.2.3 - [Error Fixed]
इसमें `torchcodec` फिक्स, Silence Remover और High-Speed इंजन शामिल है।

In [ ]:
# @title 💤 Step 1: Force Install Dependencies (जरूरी लाइब्रेरी)
import os
from IPython.display import display, Javascript

# Anti-Sleep Script
display(Javascript('''
function ClickConnect(){ document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect,60000)
'''))

print("⏳ लाइब्रेरी इंस्टॉल हो रही हैं... इसमें 1-2 मिनट लगेंगे।")
# torchcodec और TTS को साथ में इंस्टॉल करना
!pip install -q gradio librosa soundfile coqui-tts torchcodec
os.makedirs("outputs", exist_ok=True)
print("✅ इंजन तैयार है और torchcodec इंस्टॉल हो गया है!")

In [ ]:
# @title 🚀 Step 2: app.py (Realistic Voice + Silence Remover)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import re, os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

# TTS 5 (XTTS v2) Loading
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def studio_pro_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    
    # क्लीनिंग और हकलाहट फिक्स
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out_path = 'outputs/final_studio_voice.wav'
    
    # High Speed Generation
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=out_path,
        split_sentences=True
    )
    
    # Post-Processing
    y, sr = librosa.load(out_path)
    
    if sil_rem: 
        # Silence Remover
        y, _ = librosa.effects.trim(y, top_db=25)
    
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🔱 Pro Voice Studio v2.2.3')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Tags: [laugh], [sigh])', lines=8)
            smp = gr.Audio(label='Upload Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, step=0.01, label="Speed Control")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Voice Pitch")
            sil = gr.Checkbox(label="Silence Remover (सन्नाटा हटाएँ)", value=True)
            btn = gr.Button('Generate High-Speed Realistic Audio ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Final Human Touch Output')
            gr.Markdown('**टिप्स:** आवाज़ में गहराई के लिए [sigh] का प्रयोग करें।')

    btn.click(studio_pro_engine, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप स्क्रिप्ट तैयार है! अब लॉन्च हो रहा है...")
!python app.py

In [ ]:
# @title 📁 Step 3: Download Audio (डाउनलोड)
from google.colab import files
if os.path.exists('outputs/final_studio_voice.wav'):
    files.download('outputs/final_studio_voice.wav')
    print("✅ फाइल डाउनलोड हो रही है।")
else:
    print("⚠️ ऑडियो अभी बना नहीं है!")